# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smag-ev/flyrank-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row = one published web article per client (`content_id`).  
**Time Window:** A single cross-sectional snapshot utilizing up to 90 days of trailing historical search performance data prior to the decision point.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Features (The 5-Feature Frame):**
1. `content_age_days`: Knowable at the decision moment because publish dates are static and logged prior to the review window.
2. `impressions_prev_30d`: Knowable at the decision moment because trailing Google Search Console metrics are fully settled for previous periods.
3. `avg_position`: Knowable at the decision moment because daily average rankings are aggregated and finalized before the prediction occurs.
4. `word_count`: Knowable at the decision moment because content length is a fixed attribute of the currently published page.
5. `search_volume`: Knowable at the decision moment because third-party keyword search volumes are pulled prior to the prediction window.

**Label / Proxy:**
* `target_is_declining`: Derived mathematically as `1` if `trend_direction == 'down'`, else `0`.

**Context:**
* `content_id`, `client_id`: Used only to identify rows and group results, never as features for the model.

**Excluded:**
* `trend_pct` and `trend_direction`: Deliberately excluded from the feature set because they are calculated *after* the period ends. Using them as inputs causes massive feature leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Load data slice (simulating the warehouse data pull)
df = pd.read_csv("https://raw.githubusercontent.com/smag-ev/flyrank-tasks/main/data/raw/content_refresh_anonymized.csv")

# ---------------------------------------------------------
# PART 1: VERIFICATION QUERIES
# ---------------------------------------------------------

# 1. Grain Check
total_rows = len(df)
unique_ids = df['content_id'].nunique()
print(f"1. Grain Check: {total_rows} total rows, {unique_ids} unique content IDs. (1 row = 1 article)")

# 2. Row Count & Date Span
print(f"2. Row Count: {total_rows} rows.")
print(f"   Time Window: 90-day trailing snapshot max.")

# 3. Availability Check (Filtering with non-null logic)
features = ['content_age_days', 'impressions_prev_30d', 'avg_position', 'word_count', 'search_volume']
df_available = df.dropna(subset=features).copy()
print(f"3. Availability Check: {len(df_available)} rows survive after ensuring all 5 features are present.")

# ---------------------------------------------------------
# PART 2: THE FEATURE LEAKAGE TRAP
# ---------------------------------------------------------
print("\n" + "="*40)
print("EXPERIMENT: SPRINGING THE LEAKAGE TRAP")
print("="*40)

# Create the true label
df_available['target_is_declining'] = (df_available['trend_direction'] == 'down').astype(int)

# THE TRAP: We intentionally include 'trend_pct' (a future-derived column) as a feature
X_trap = df_available[features + ['trend_pct']]
y = df_available['target_is_declining']

# Train/Test Split
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_trap, y, test_size=0.2, random_state=42)

# Train a quick model on leaked data
model_trap = RandomForestClassifier(random_state=42, max_depth=3)
model_trap.fit(X_train_t, y_train_t)
trap_preds = model_trap.predict(X_test_t)

print(f"Trap Score (with leaked 'trend_pct'): {precision_score(y_test_t, trap_preds):.1%} Precision")
print("-> Dangerously perfect! The model 'cheated' by looking at the future outcome.\n")

print("REMOVING THE TRAP (HONEST EVALUATION)")
# Remove the leaked column
X_honest = df_available[features]
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)

# Train on honest features only
model_honest = RandomForestClassifier(random_state=42, max_depth=3)
model_honest.fit(X_train_h, y_train_h)
honest_preds = model_honest.predict(X_test_h)

print(f"Honest Score (using only 5 allowed features): {precision_score(y_test_h, honest_preds):.1%} Precision")
print("-> This is the real baseline we will try to beat during modeling weeks.")


1. Grain Check: 30000 total rows, 30000 unique content IDs. (1 row = 1 article)
2. Row Count: 30000 rows.
   Time Window: 90-day trailing snapshot max.
3. Availability Check: 20018 rows survive after ensuring all 5 features are present.

EXPERIMENT: SPRINGING THE LEAKAGE TRAP
Trap Score (with leaked 'trend_pct'): 100.0% Precision
-> Dangerously perfect! The model 'cheated' by looking at the future outcome.

REMOVING THE TRAP (HONEST EVALUATION)
Honest Score (using only 5 allowed features): 67.3% Precision
-> This is the real baseline we will try to beat during modeling weeks.


## 4. Data limits

**Limitation:** Blindness to off-page external shocks.
This dataset relies purely on historical Google Search Console metrics (impressions, clicks, average position) and on-page attributes (word count, age). It is completely blind to external market forces that dictate search traffic—such as a competitor launching a massive PR campaign or Google rolling out a sudden core algorithm update.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.